# SRF for BME — Experiments Notebook

End-to-end experimental study of the proposed Stabilization Reserve Fund (SRF). All
results in Section 4 of the paper are produced by this notebook. It depends only
on the in-tree `srf_lab` package and standard scientific-Python libraries.

**Outline**

1. Setup
2. Performance: BME with vs. without SRF (synthetic crash)
3. Effect of the dynamic tax (with vs. without)
4. Baseline ablation (Table 4)
5. BME organic-flux scenarios (Table 3)
6. Market-impact comparison: linear vs AMM, with liquidity shock
7. Sensitivity heatmap over $(\omega_p, \omega_d)$
8. Failure boundary under liquidity-shock severity / reserve fraction
9. Save tables and figures

## 1. Setup

In [ ]:
import os, sys, json
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the in-tree package importable
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)

from srf_lab import (
    Config, simulate, monte_carlo,
    NoController, PassiveThresholdController, ProportionalController,
    PDController, LockOnlyController, SRFNoTaxController, FullSRFController,
    LinearImpact, AMMImpact, LiquidityShock,
    BMEScenario, bme_flux,
    compute_metrics,
)
from srf_lab.bme import make_scenario, list_scenarios
from srf_lab.market_impact import make_impact

FIG_DIR = os.path.join(ROOT, 'results', 'figs');  os.makedirs(FIG_DIR, exist_ok=True)
TBL_DIR = os.path.join(ROOT, 'results', 'tables'); os.makedirs(TBL_DIR, exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 200, 'figure.figsize': (8, 4.5)})
PALETTE = {
    'baseline':    '#9aa0a6',
    'passive':     '#bb6588',
    'p_only':      '#e9c46a',
    'pd':          '#f4a261',
    'lock_only':   '#9b5de5',
    'srf_no_tax':  '#264653',
    'full_srf':    '#2a9d8f',
    'crash':       '#e76f51',
}
print('CWD:', os.getcwd())
print('ROOT:', ROOT)

## 2. Performance: BME with vs without SRF (synthetic crash)

Reproduces the headline result from the original notebook: a flash-crash in days 150–160 against an unregulated baseline.

In [ ]:
cfg_basic = Config(T=365, seed=42, stress_kind='gbm_crash', bme_scenario='neutral')

ctrl_none = NoController(R0=cfg_basic.rho_0 * cfg_basic.P0 * cfg_basic.S0)
ctrl_srf  = FullSRFController(R0=cfg_basic.rho_0 * cfg_basic.P0 * cfg_basic.S0)

traj_base = simulate(cfg_basic, ctrl_none)
traj_srf  = simulate(cfg_basic, ctrl_srf)

m_base = compute_metrics(traj_base)
m_srf  = compute_metrics(traj_srf, baseline_traj=traj_base)

summary = pd.DataFrame({
    'No intervention': m_base,
    'Full SRF':        m_srf,
}).T
summary

In [ ]:
# ---- Figure 4: price trajectory with vs without SRF ----
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(9, 8.5),
                                      gridspec_kw={'height_ratios': [3, 1, 1]}, sharex=True)
t = np.arange(cfg_basic.T)
ax1.plot(t, traj_base.P, label='No intervention', color=PALETTE['baseline'], linestyle='--', alpha=0.85)
ax1.plot(t, traj_srf.P,  label='Full SRF (PD + lock + tax)', color=PALETTE['full_srf'], lw=2)
ax1.plot(t, traj_srf.P_target, label='EMA target', color='#264653', linestyle=':', alpha=0.7)
ax1.axvspan(150, 160, color=PALETTE['crash'], alpha=0.10, label='Engineered crash')
ax1.set_ylabel('Price (USD)'); ax1.legend(loc='lower left'); ax1.set_title('Figure 4. Price trajectory under engineered crash')

ax2.plot(t, traj_srf.R / 1e6, color='#9b5de5', lw=1.6, label='Reserve $R_t$ (M USD)')
ax2.fill_between(t, traj_srf.R/1e6, color='#9b5de5', alpha=0.20)
ax2.set_ylabel('Reserve (M USD)'); ax2.legend(loc='upper left')

ax3.plot(t, traj_srf.lock_ratio*100, color='#f4a261', lw=1.6, label='Lock ratio $\\ell_t$ (%)')
ax3.plot(t, traj_srf.eta*100, color='#2a9d8f', lw=1.4, label='Tax rate $\\eta_t$ (%)')
ax3.set_ylabel('% '); ax3.set_xlabel('day'); ax3.legend(loc='upper left')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig4_price_trajectory.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'fig4_price_trajectory.png'), bbox_inches='tight')
plt.show()

## 3. Effect of the dynamic tax — with and without

Compares `SRFNoTaxController` (PD + lock, no tax) against `FullSRFController`.

In [ ]:
ctrl_no_tax = SRFNoTaxController(R0=cfg_basic.rho_0 * cfg_basic.P0 * cfg_basic.S0)
ctrl_full   = FullSRFController(R0=cfg_basic.rho_0 * cfg_basic.P0 * cfg_basic.S0)
traj_no_tax = simulate(cfg_basic, ctrl_no_tax)
traj_full   = simulate(cfg_basic, ctrl_full)

tab_tax = pd.DataFrame({
    'No SRF':           compute_metrics(traj_base),
    'SRF (no tax)':     compute_metrics(traj_no_tax, baseline_traj=traj_base),
    'Full SRF (+tax)':  compute_metrics(traj_full,   baseline_traj=traj_base),
}).T
display(tab_tax)

# ---- Figure 5: reserve balance and tax rate ----
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
a1.plot(t, traj_no_tax.R/1e6, label='SRF (no dynamic tax)',  color='#264653', lw=1.6)
a1.plot(t, traj_full.R/1e6,   label='Full SRF (+ dynamic tax)', color='#2a9d8f', lw=1.8)
a1.axhline(traj_full.R[0]/1e6, color='gray', linestyle=':', alpha=0.6, label='Initial $R_0$')
a1.set_ylabel('Reserve (M USD)'); a1.legend(loc='lower left'); a1.set_title('Figure 5. Reserve balance and dynamic tax rate')

a2.plot(t, traj_full.eta*100, color='#2a9d8f', lw=1.6, label='$\\eta_t$ (Full SRF)')
a2.fill_between(t, traj_full.eta*100, color='#2a9d8f', alpha=0.20)
a2.set_ylabel('Tax rate (%)'); a2.set_xlabel('day'); a2.legend(loc='upper right')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig5_reserve_tax.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'fig5_reserve_tax.png'), bbox_inches='tight')
plt.show()

## 4. Baseline ablation — Table 4

All seven controllers under the same engineered-crash scenario, averaged over
`N_RUNS` Monte-Carlo seeds.

In [ ]:
from srf_lab.controllers import _ControllerBase

N_RUNS = 30
cfg_mc = Config(T=365, seed=42, stress_kind='gbm_crash', bme_scenario='neutral')

controllers = {
    'No intervention':         (NoController,                {}),
    'Passive threshold':        (PassiveThresholdController,  {}),
    'P-only':                   (ProportionalController,      {}),
    'PD':                       (PDController,                {}),
    'Lock-only':                (LockOnlyController,          {}),
    'SRF (no tax)':             (SRFNoTaxController,          {}),
    'Full SRF':                 (FullSRFController,           {}),
}

abl_rows = []
for label, (cls, kw) in controllers.items():
    df = monte_carlo(cfg_mc, cls, n_runs=N_RUNS, controller_kwargs=kw)
    row = df.mean(numeric_only=True).to_dict()
    row['controller'] = label
    abl_rows.append(row)

abl_table = pd.DataFrame(abl_rows).set_index('controller')
abl_keep = ['ann_vol', 'max_drawdown', 'downside_vol', 'expected_shortfall_5',
             'time_under_dd_10pct', 'reserve_depletion', 'avg_lock', 'peak_lock',
             'avg_tax', 'intervention_cost', 'n_interventions']
abl_table[abl_keep].round(4)

## 5. BME organic-flux scenarios — Table 3

Compares the **No intervention** baseline against **Full SRF** on the five canonical BME regimes (no liquidity shock).

In [ ]:
scenarios = ['neutral', 'emission_heavy', 'burn_heavy', 'reward_inflation', 'demand_collapse']
scenario_labels = {
    'neutral':           'Neutral',
    'emission_heavy':    'Emission-heavy',
    'burn_heavy':        'Burn-heavy',
    'reward_inflation':  'Reward-driven inflation',
    'demand_collapse':   'Demand collapse',
}

rows = []
for sc in scenarios:
    cfg_sc = replace(cfg_mc, bme_scenario=sc)
    base_df = monte_carlo(cfg_sc, NoController,         n_runs=N_RUNS)
    full_df = monte_carlo(cfg_sc, FullSRFController,    n_runs=N_RUNS)
    rows.append({
        'scenario':           scenario_labels[sc],
        'base_ann_vol':       base_df['ann_vol'].mean(),
        'base_max_dd':        base_df['max_drawdown'].mean(),
        'srf_ann_vol':        full_df['ann_vol'].mean(),
        'srf_max_dd':         full_df['max_drawdown'].mean(),
        'vol_reduction':      full_df['vol_reduction'].mean() if 'vol_reduction' in full_df else float('nan'),
        'dd_reduction':       full_df['dd_reduction'].mean() if 'dd_reduction' in full_df else float('nan'),
        'reserve_depletion':  full_df['reserve_depletion'].mean(),
        'peak_lock':          full_df['peak_lock'].mean(),
    })
scen_table = pd.DataFrame(rows).set_index('scenario')
scen_table.round(4)

## 6. Market-impact comparison — linear vs AMM, with liquidity shock

Holds the controller fixed (Full SRF) and varies the impact model + liquidity shock severity. Generates Figure 8 (failure boundary).

In [ ]:
impact_rows = []
for impact_kind in ['linear', 'amm']:
    for liq_shock in [False, True]:
        cfg_im = replace(cfg_mc, impact_model=impact_kind, liq_shock=liq_shock,
                          liq_shock_t=148, liq_shock_len=20, liq_shock_drop=0.7)
        df = monte_carlo(cfg_im, FullSRFController, n_runs=N_RUNS)
        row = df.mean(numeric_only=True).to_dict()
        row['impact_model'] = impact_kind
        row['liquidity_shock'] = liq_shock
        impact_rows.append(row)
impact_table = pd.DataFrame(impact_rows).set_index(['impact_model', 'liquidity_shock'])
impact_table[['ann_vol', 'max_drawdown', 'reserve_depletion', 'peak_lock', 'intervention_cost']].round(4)

## 7. Sensitivity heatmap over $(\omega_p, \omega_d)$ — Figure 7

Sweeps the proportional and derivative gains and reports the average max-drawdown over Monte-Carlo seeds. Helps justify the chosen operating point.

In [ ]:
omega_p_grid = np.array([0.5, 0.75, 1.0, 1.5, 2.0, 3.0])
omega_d_grid = np.array([0.5, 1.0, 1.5, 2.0, 3.0, 4.0])
MC_REDUCED = 12   # reduce MC count for the sweep

Z = np.zeros((len(omega_p_grid), len(omega_d_grid)))
for i, wp in enumerate(omega_p_grid):
    for j, wd in enumerate(omega_d_grid):
        cfg_g = replace(cfg_mc, omega_p=float(wp), omega_d=float(wd))
        df = monte_carlo(cfg_g, FullSRFController, n_runs=MC_REDUCED)
        Z[i, j] = df['max_drawdown'].mean()

fig, ax = plt.subplots(figsize=(7.5, 5.5))
im = ax.imshow(Z*100, origin='lower', aspect='auto', cmap='RdYlGn_r',
                extent=[omega_d_grid[0]-0.25, omega_d_grid[-1]+0.25,
                        omega_p_grid[0]-0.125, omega_p_grid[-1]+0.125])
ax.set_xticks(omega_d_grid); ax.set_yticks(omega_p_grid)
ax.set_xlabel('$\\omega_d$ (derivative gain)')
ax.set_ylabel('$\\omega_p$ (proportional gain)')
ax.set_title('Figure 7. Sensitivity surface — max drawdown (%) by $(\\omega_p, \\omega_d)$')
for i in range(len(omega_p_grid)):
    for j in range(len(omega_d_grid)):
        ax.text(omega_d_grid[j], omega_p_grid[i], f'{Z[i,j]*100:.1f}',
                ha='center', va='center', color='black', fontsize=9)
fig.colorbar(im, ax=ax, label='Max drawdown (%)')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig7_sensitivity.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'fig7_sensitivity.png'), bbox_inches='tight')
plt.show()

## 8. Failure boundary under liquidity shock & reserve fraction — Figure 8

In [ ]:
drop_grid = np.linspace(0.0, 0.9, 6)        # liquidity drop severity
rho_grid  = np.array([0.05, 0.075, 0.10, 0.15, 0.20])  # initial reserve fraction
MC_FAIL = 8

Z2 = np.zeros((len(rho_grid), len(drop_grid)))
for i, rho in enumerate(rho_grid):
    for j, drop in enumerate(drop_grid):
        cfg_f = replace(cfg_mc, rho_0=float(rho), liq_shock=True,
                         liq_shock_t=148, liq_shock_len=20,
                         liq_shock_drop=float(drop))
        df = monte_carlo(cfg_f, FullSRFController, n_runs=MC_FAIL)
        Z2[i, j] = df['max_drawdown'].mean()

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(Z2*100, origin='lower', aspect='auto', cmap='RdYlGn_r',
                extent=[drop_grid[0]-0.05, drop_grid[-1]+0.05,
                        rho_grid[0]-0.0125, rho_grid[-1]+0.0125])
ax.set_xticks(drop_grid); ax.set_yticks(rho_grid)
ax.set_xlabel('Liquidity-shock severity (drop fraction)')
ax.set_ylabel('Initial reserve fraction $\\rho_0$')
ax.set_title('Figure 8. Failure boundary — max drawdown under liquidity shock')
for i in range(len(rho_grid)):
    for j in range(len(drop_grid)):
        ax.text(drop_grid[j], rho_grid[i], f'{Z2[i,j]*100:.1f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax, label='Max drawdown (%)')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig8_failure_boundary.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'fig8_failure_boundary.png'), bbox_inches='tight')
plt.show()

## 9. Lock ratio + effective supply — Figure 6

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
a1.plot(t, traj_full.lock_ratio*100, color='#9b5de5', lw=1.7, label='Lock ratio $\\ell_t$ (%)')
a1.fill_between(t, traj_full.lock_ratio*100, color='#9b5de5', alpha=0.2)
a1.set_ylabel('Lock ratio (%)'); a1.legend(loc='upper left'); a1.set_title('Figure 6. Volatility-triggered lock ratio and effective supply')

S_eff = traj_full.S * (1 - traj_full.lock_ratio)
a2.plot(t, traj_full.S/1e6,    label='Total supply $S_t$', color='#264653', linestyle='--')
a2.plot(t, S_eff/1e6,          label='Effective supply $S_{\\mathrm{eff},t}=S_t(1-\\ell_t)$', color='#2a9d8f', lw=1.8)
a2.set_ylabel('Supply (M tokens)'); a2.set_xlabel('day'); a2.legend(loc='lower left')
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig6_lock_supply.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'fig6_lock_supply.png'), bbox_inches='tight')
plt.show()

## 10. Save tables (CSV + LaTeX)

In [ ]:
def save_table(df, stem, caption, label, **fmt):
    df.to_csv(os.path.join(TBL_DIR, stem + '.csv'))
    df.to_latex(os.path.join(TBL_DIR, stem + '.tex'),
                 caption=caption, label=label,
                 float_format=lambda x: f'{x:.4f}', **fmt)

save_table(abl_table[abl_keep],   'table4_baseline_ablation',
            'Baseline ablation: per-controller metrics under engineered crash (mean over 30 seeds).',
            'tab:abl')
save_table(scen_table,             'table3_bme_scenarios',
            'BME organic-flux scenarios (mean over 30 seeds).',
            'tab:bme')
save_table(impact_table,           'table_impact_models',
            'Impact-model comparison with optional liquidity shock.',
            'tab:impact')
save_table(summary,                'table2_main_result',
            'Main result: No intervention vs Full SRF on the engineered-crash scenario.',
            'tab:main')
print('All tables saved to', TBL_DIR)